# 評価と図示 (Transformer vs LSTM)

In [ ]:
import os, sys, pickle
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.evaluate import compute_bleu, bleu_by_length

with open('../results/history_transformer.pkl', 'rb') as f:
    hist_t = pickle.load(f)
with open('../results/history_lstm.pkl', 'rb') as f:
    hist_l = pickle.load(f)
with open('../results/translations_test.pkl', 'rb') as f:
    trans = pickle.load(f)

## 学習曲線 (val loss)

In [ ]:
plt.plot(hist_t['val_loss'], label='Transformer')
plt.plot(hist_l['val_loss'], label='LSTM', linestyle='--')
plt.xlabel('epoch'); plt.ylabel('val loss'); plt.legend()
plt.savefig('../results/loss_curve.png')
plt.show()

## BLEU

In [ ]:
bleu_t = compute_bleu(trans['transformer'], trans['refs'])
bleu_l = compute_bleu(trans['lstm'],        trans['refs'])
print(f'Transformer: {bleu_t:.2f}')
print(f'LSTM:        {bleu_l:.2f}')

## 文長別 BLEU

In [ ]:
by_len_t = bleu_by_length(trans['transformer'], trans['refs'], trans['src_lens'])
by_len_l = bleu_by_length(trans['lstm'],        trans['refs'], trans['src_lens'])
labels = [r['range'] for r in by_len_t]
x = np.arange(len(labels))

plt.bar(x - 0.2, [r['bleu'] or 0 for r in by_len_t], 0.4, label='Transformer')
plt.bar(x + 0.2, [r['bleu'] or 0 for r in by_len_l], 0.4, label='LSTM')
plt.xticks(x, labels); plt.xlabel('source length'); plt.ylabel('BLEU'); plt.legend()
plt.savefig('../results/bleu_by_length.png')
plt.show()

for t, l in zip(by_len_t, by_len_l):
    print(f"{t['range']}: T={t['bleu']:.2f} L={l['bleu']:.2f}")

## 訳例

In [ ]:
for i in [0, 1, 2, 10, 50]:
    print(f'--- sample {i} ---')
    print('REF :', trans['refs'][i])
    print('TRA :', trans['transformer'][i])
    print('LSTM:', trans['lstm'][i])
    print()